In [0]:
from pyspark.sql.functions import when
import pyspark.sql.types as T
df = spark.table("workspace.bronze.customer_survey")
df = df.drop("_file", "_line", "_modified", "_fivetran_synced")

target_schema = T.StructType([
    T.StructField("survey_id", T.StringType()),   
    T.StructField("order_id", T.StringType(), True),
    T.StructField("survey_sent_date", T.DateType(), True),
    T.StructField("survey_response_date", T.DateType(), True)   ,
    T.StructField("responded_flag", T.BooleanType(), True),
    T.StructField("delivered_on_time_rating", T.IntegerType(), True),
    T.StructField("work_quality_rating", T.IntegerType(), True),
    T.StructField("cleanliness_rating", T.IntegerType(), True),
    T.StructField("communication_rating", T.IntegerType(), True),
    T.StructField("overall_satisfaction_rating", T.IntegerType() , True)
])
#reconciled
df = df.to(target_schema)
display(df)
df = df.withColumn("delivered_on_time_rating",when(df["delivered_on_time_rating"].isNull(), 0).otherwise(df["delivered_on_time_rating"])) \
       .withColumn("work_quality_rating", when(df["work_quality_rating"].isNull(), 0).otherwise(df["work_quality_rating"])) \
       .withColumn("cleanliness_rating", when(df["cleanliness_rating"].isNull(), 0).otherwise(df["cleanliness_rating"])) \
       .withColumn("communication_rating", when(df["communication_rating"].isNull(), 0).otherwise(df["communication_rating"])) \
       .withColumn("overall_satisfaction_rating", when(df["overall_satisfaction_rating"].isNull(),0).otherwise(df["overall_satisfaction_rating"]))
df.write.mode("overwrite").saveAsTable("workspace.silver.customer_survey")

In [0]:
from pyspark.sql import functions as F
df = spark.table("workspace.bronze.order")
df = df.drop("_file", "_line", "_modified", "_fivetran_synced")
df = df.withColumn("technician_id", when(df["technician_id"].isNull(), "UNKNOWN").otherwise(df["technician_id"])) 
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.order")

In [0]:
df = spark.table("workspace.bronze.store")
df = df.drop("_file", "_line", "_modified", "_fivetran_synced")
df = df.withColumn(
    "manager_name",
    when(df["manager_name"].isNull(),"UNKNOWN" ).otherwise(df["manager_name"])
)
df.write.mode("overwrite").saveAsTable("workspace.silver.store")

In [0]:
df = spark.table("workspace.bronze.estimate")
df = df.drop("_file", "_line", "_modified", "_fivetran_synced")
df = df.withColumn(
    "estimate_amount",
    when(df["estimate_amount"].isNull(),0 ).otherwise(df["estimate_amount"])
)
df.write.mode("overwrite").saveAsTable("workspace.silver.estimate")

In [0]:
df = spark.table("workspace.bronze.invoice")
df = df.drop("_file", "_line", "_modified", "_fivetran_synced")
df = df.withColumn(
    "payment_mode",
    when(df["payment_mode"].isNull(),"Unknown" ).otherwise(df["payment_mode"])
)
df.write.mode("overwrite").saveAsTable("workspace.silver.invoice")

In [0]:
df = spark.table("workspace.bronze.ns_budget")
df = df.drop("_file", "_line", "_modified", "_fivetran_synced")
df.write.mode("overwrite").saveAsTable("workspace.silver.ns_budget")
